# Advanced Content Based Recommender

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
cleaned_df = pd.read_csv("../data/cleaned_amazon.csv")

In [3]:
recommendation_df = (
    cleaned_df
    .drop_duplicates(
        subset='product_name'
    )
    .reset_index(drop=True)
)

In [4]:
recommendation_df.shape

(1337, 19)

In [5]:
recommendation_df['combined_features'] = (
    recommendation_df['category']
    + " "
    +
    recommendation_df['about_product']
)

In [6]:
tfidf = TfidfVectorizer(
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(
    recommendation_df['combined_features']
)

In [7]:
tfidf_matrix.shape

(1337, 9163)

In [8]:
cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

In [9]:
cosine_sim.shape

(1337, 1337)

In [10]:
indices = pd.Series(
    recommendation_df.index,
    index=recommendation_df['product_name']
)

In [11]:
indices.head()

product_name
Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)                                         0
Ambrane Unbreakable 60W / 3A Fast Charging 1.5m Braided Type C Cable for Smartphones, Tablets, Laptops & other Type C devices, PD Technology, 480Mbps Data Sync, Quick Charge 3.0 (RCT15A, Black)          1
Sounce Fast Phone Charging Cable & Data Sync USB Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini & iOS Devices                                                                   2
boAt Deuce USB 300 2 in 1 Type-C & Micro USB Stress Resistant, Tangle-Free, Sturdy Cable with 3A Fast Charging & 480mbps Data Transmission, 10000+ Bends Lifespan and Extended 1.5m Length(Martian Red)    3
Portronics Konnect L 1.2M Fast Charging 3A 8 Pin USB Cable with Charge & Sync Function for iPhone, iPad (Grey)                                                         

In [12]:
def recommend_products(product_name,
                       top_n=10):

    idx = indices[product_name]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    product_indices = [
        i[0]
        for i in similarity_scores
    ]

    scores = [
        i[1]
        for i in similarity_scores
    ]

    recommendations = recommendation_df.iloc[
        product_indices
    ][
        ['product_name',
         'rating',
         'rating_count']
    ]

    recommendations['similarity_score'] = scores

    return recommendations

In [15]:
recommend_products(
    recommendation_df[
        'product_name'
    ].iloc[0]
)

,product_name,rating,rating_count,similarity_score
220,Wayona Nylon Braided Usb Syncing And Charging ...,4.2,24269.0,0.975083
42,Wayona Nylon Braided 3A Lightning to USB A Syn...,4.2,24269.0,0.937440
89,Wayona Nylon Braided (2 Pack) Lightning Fast U...,4.2,24269.0,0.937440
166,Wayona Nylon Braided Lightning USB Data Sync &...,4.2,13120.0,0.771227
80,Wayona Usb Nylon Braided Data Sync And Chargin...,4.2,24269.0,0.755197
106,Wayona Nylon Braided 2M / 6Ft Fast Charge Usb ...,4.2,24269.0,0.705136
208,MYVN LTG to USB for Fast Charging & Data Sync ...,3.7,2249.0,0.594546
104,Wayona Nylon Braided USB Data Sync and Fast Ch...,4.2,13120.0,0.471609
261,Wayona Nylon Braided Usb Type C 3Ft 1M 3A Fast...,4.2,10576.0,0.471387
322,"REDTECH USB-C to Lightning Cable 3.3FT, [Apple...",5.0,5179.0,0.437026


In [18]:
recommend_products(
    recommendation_df['product_name'].iloc[50]
)

,product_name,rating,rating_count,similarity_score
141,TP-LINK AC1300 Archer T3U Plus High Gain USB 3...,4.4,24780.0,0.766039
598,TP-Link Archer AC1200 Archer C6 Wi-Fi Speed Up...,4.4,35024.0,0.443309
143,TP-Link Nano USB WiFi Dongle 150Mbps High Gain...,4.2,179692.0,0.437825
56,TP-LINK WiFi Dongle 300 Mbps Mini Wireless Net...,4.2,179691.0,0.401325
844,"TP-Link AC1200 Archer A6 Smart WiFi, 5GHz Giga...",4.4,12679.0,0.393914
170,TP-Link AC1300 USB WiFi Adapter (Archer T3U) -...,4.4,23169.0,0.392191
671,"TP-Link AC750 Dual Band Wireless Cable Router,...",4.3,68409.0,0.382587
876,TP-Link TL-WA855RE 300 Mbps Wi-Fi Range Extend...,4.2,16182.0,0.360814
98,TP-Link UE300 USB 3.0 to RJ45 Gigabit Ethernet...,4.5,22420.0,0.338882
43,TP-Link Nano AC600 USB Wi-Fi Adapter(Archer T2...,4.3,12093.0,0.325871


In [19]:
def search_products(keyword):

    results = recommendation_df[
        recommendation_df['product_name']
        .str.contains(
            keyword,
            case=False,
            na=False
        )
    ]

    return results[['product_name']].head(20)

In [20]:
search_products("samsung")

,product_name
22,Samsung 80 cm (32 Inches) Wondertainment Serie...
32,Zoul USB C 60W Fast Charging 3A 6ft/2M Long Ty...
33,Samsung Original Type C to C Cable - 3.28 Feet...
48,7SEVEN® Compatible for Samsung Smart 4K Ultra ...
61,Samsung 108 cm (43 inches) Crystal 4K Neo Seri...
70,oraimo 65W Type C to C Fast Charging Cable USB...
84,Wayona Usb Type C Fast Charger Cable Fast Char...
87,Samsung 108 cm (43 inches) Crystal 4K Series U...
102,Isoelite Remote Compatible for Samsung LED/LCD...
105,Wayona Type C To Type C Long Fast Charging Cab...


In [21]:
search_products("boat")

,product_name
3,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...
6,"boAt Micro USB 55 Tangle-free, Sturdy Micro US..."
11,boAt Rugged v3 Extra Tough Unbreakable Braided...
18,"boAt Type C A325 Tangle-free, Sturdy Type C Ca..."
29,boAt A400 USB Type-C to USB-A 2.0 Male Data Ca...
74,"boAt Type C A750 Stress Resistant, Tangle-free..."
83,"boAt A 350 Type C Cable for Smartphone, Chargi..."
92,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...
111,"boAt Laptop, Smartphone Type-c A400 Male Data ..."
113,"boAt Type C A750 Stress Resistant, Tangle-free..."


In [22]:
recommendation_df.to_csv(
    "../data/recommendation_dataset.csv",
    index=False
)